# Урок 27 — Алгоритми сортування: від Big-O до реального заліза

**Модуль 3 · Python Advanced · Автор: Viktor Nikoriak**

---

## Про що цей урок?

Сортування — це не просто набір алгоритмів із різною складністю.

Це **точка перетину трьох рівнів системи**:

* абстрактна математика (Big-O),
* структура даних,
* фізична поведінка пам’яті та процесора.

Одна і та сама асимптотика може давати **кардинально різний результат на практиці**, тому що:

* доступ до пам’яті дорожчий за арифметику,
* CPU кеш визначає швидкість більше, ніж формула,
* копіювання даних іноді важливіше, ніж кількість порівнянь,
* рекурсія має реальну ціну.

Цей урок — не про “вивчити сортування”.

Це про те, щоб зрозуміти:

> як алгоритм поводиться як **система**, а не як формула.

---

## Структура уроку

| Частина | Тема                                                  |
| ------- | ----------------------------------------------------- |
| 1       | Bubble Sort — наочний старт та оптимізація            |
| 2       | Selection Sort — мінімум записів у пам'ять            |
| 3       | Insertion Sort — чемпіон майже відсортованих даних    |
| 4       | Merge Sort — гарантований $O(n \log n)$ ціною пам'яті |
| 5       | Heap Sort — сортування через структуру даних          |
| 6       | Quick Sort — король кешу та практики                  |
| 7       | Timsort — секретна зброя Python                       |
| 8       | Апаратна реальність: кеш, пам'ять, рекурсія           |
| 9       | Benchmark — вимірюємо час всіх алгоритмів             |
| 10      | Підсумок                                              |

---


---

## Частина 1: Bubble Sort — бульбашкове сортування

---

### Інтуїція

Алгоритм проходить по списку, порівнює сусідні елементи і міняє їх місцями, якщо вони в неправильному порядку. Більші елементи «спливають» вгору (до кінця списку) як бульбашки у воді — звідси назва.

```
Прохід 1:  [5, 2, 1, 8, 4]
           [2, 5, 1, 8, 4]   ← 5 > 2, swap
           [2, 1, 5, 8, 4]   ← 5 > 1, swap
           [2, 1, 5, 8, 4]   ← 5 < 8, no swap
           [2, 1, 5, 4, 8]   ← 8 > 4, swap  →  8 «спливла» на місце!
```

### Складність

| Випадок | Складність | Причина |
|---------|-----------|--------|
| **Найкращий** | $O(n)$ | Масив вже відсортований — оптимізований алгоритм робить лише 1 прохід |
| **Середній** | $O(n^2)$ | Подвійний цикл порівнянь |
| **Найгірший** | $O(n^2)$ | Зворотний порядок — максимум swap для кожного елемента |

### Апаратна реальність

- **Cache locality:** Висока — працює з сусідніми елементами, добре потрапляє в кеш-лінії CPU
- **Swaps:** Дуже багато — найдорожча частина; кожен swap це 3 операції запису в пам'ять
- **Стабільність:** Стабільний (однакові елементи не міняються місцями)
- **Пам'ять:** $O(1)$ — in-place

### Реальне застосування

**Ніде в продакшені.** Barack Obama на інтерв'ю Google: *"Bubble Sort — the wrong way to go."* Використовується виключно для навчання концепції порівняння та обміну.

---

In [ ]:
# ── Bubble Sort: базова та оптимізована реалізації ────────────────────────────

def bubble_sort_basic(arr):
    """
    Базовий Bubble Sort: завжди робить n*(n-1)/2 порівнянь.
    Навіть якщо масив вже відсортований — продовжує непотрібні проходи.
    Складність: O(n^2) у всіх випадках.
    """
    n = len(arr)
    for i in range(n):               # зовнішній цикл: n проходів
        for j in range(n - 1):       # внутрішній: порівнює сусідів
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]  # swap — 3 операції пам'яті!
    return arr


def bubble_sort(arr):
    """
    Оптимізований Bubble Sort з двома поліпшеннями:
    1. Прапорець swapped: зупиняємось, якщо прохід без swap → O(n) для відсортованого.
    2. Кожен прохід гарантовано ставить найбільший елемент на місце → скорочуємо межу.
    """
    n = len(arr)
    for i in range(n):
        swapped = False                     # прапорець: чи були swap у цьому проході?
        for j in range(n - 1 - i):         # -i: кінець масиву вже відсортований
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swapped = True
        if not swapped:                     # жодного swap → масив відсортовано!
            break
    return arr


# ── Демонстрація ──────────────────────────────────────────────────────────────
print("=== Bubble Sort ===")
data = [5, 2, 1, 8, 4]
print(f"  Вхід:            {data}")
result = bubble_sort(data.copy())
print(f"  Відсортовано:    {result}")

# Найкращий випадок: вже відсортований масив
already_sorted = [1, 2, 3, 4, 5]
# Порахуємо кількість проходів вручну
passes = 0
arr = already_sorted.copy()
n = len(arr)
for i in range(n):
    passes += 1
    swapped = False
    for j in range(n - 1 - i):
        if arr[j] > arr[j + 1]:
            arr[j], arr[j + 1] = arr[j + 1], arr[j]
            swapped = True
    if not swapped:
        break
print(f"\n  Вже відсортований {already_sorted}: лише {passes} прохід (O(n) !)")
print(f"  Без оптимізації: потрібно {n} проходів")

---

## Частина 2: Selection Sort — сортування вибором

---

### Інтуїція

Ділимо масив на дві частини: відсортовану (ліворуч) і невідсортовану (праворуч). На кожному кроці **знаходимо мінімальний елемент** у невідсортованій частині і переміщуємо його в кінець відсортованої.

```
[12,  8,  3, 20, 11]   ← знаходимо мін = 3 (індекс 2)
[ 3,  8, 12, 20, 11]   ← swap(0, 2), відсортовано: [3]
[ 3,  8, 12, 20, 11]   ← мін = 8 (вже на місці), відсортовано: [3, 8]
[ 3,  8, 11, 20, 12]   ← swap(2, 4), відсортовано: [3, 8, 11]
[ 3,  8, 11, 12, 20]   ← swap(3, 4), відсортовано: [3, 8, 11, 12]
                          готово!
```

### Складність

| Випадок | Складність | Причина |
|---------|-----------|--------|
| **Всі** | $O(n^2)$ | Подвійний цикл: зовнішній $n$, внутрішній від $n-1$ до $1$ |

### Апаратна реальність

- **Swaps:** Мінімальна кількість — рівно $n-1$ swap (в Bubble Sort їх може бути $O(n^2)$). Якщо операція swap дорога (наприклад, запис на диск), Selection Sort виграє у Bubble Sort
- **Порівнянь:** $\frac{n(n-1)}{2}$ — завжди, незалежно від стану масиву
- **Стабільність:** **Нестабільне** — далекі swap можуть порушити порядок однакових елементів
- **Пам'ять:** $O(1)$ — in-place

### Реальне застосування

Там, де вартість swap значно вища за вартість порівняння (наприклад, робота зі структурами великого розміру на диску). В іншому — так само навчальний алгоритм.

---

In [ ]:
# ── Selection Sort ─────────────────────────────────────────────────────────────

def selection_sort(arr):
    """
    Selection Sort: на кожному кроці знаходимо мінімум у правій частині
    і swap його до лівої (відсортованої) зони.
    Завжди рівно (n-1) swap — мінімум серед усіх алгоритмів.
    """
    n = len(arr)
    for i in range(n):                     # i — права межа відсортованої зони
        min_idx = i                         # припускаємо: перший невідсортований — мін
        for j in range(i + 1, n):           # шукаємо справжній мінімум
            if arr[j] < arr[min_idx]:
                min_idx = j
        if min_idx != i:                    # swap лише якщо знайдено менший елемент
            arr[i], arr[min_idx] = arr[min_idx], arr[i]   # 1 swap за ітерацію!
    return arr


# ── Демонстрація ──────────────────────────────────────────────────────────────
print("=== Selection Sort ===")
data = [12, 8, 3, 20, 11]
print(f"  Вхід:            {data}")
result = selection_sort(data.copy())
print(f"  Відсортовано:    {result}")

# Порівнюємо кількість swap з Bubble Sort
def count_swaps_bubble(arr):
    """Рахує кількість swap у Bubble Sort."""
    arr = arr.copy()
    swaps = 0
    n = len(arr)
    for i in range(n):
        for j in range(n - 1 - i):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swaps += 1
    return swaps

def count_swaps_selection(arr):
    """Рахує кількість swap у Selection Sort."""
    arr = arr.copy()
    swaps = 0
    n = len(arr)
    for i in range(n):
        min_idx = i
        for j in range(i + 1, n):
            if arr[j] < arr[min_idx]:
                min_idx = j
        if min_idx != i:
            arr[i], arr[min_idx] = arr[min_idx], arr[i]
            swaps += 1
    return swaps

import random
random.seed(42)
test_arr = random.sample(range(100), 20)
print(f"\n=== Порівняння кількості swap для масиву з 20 елементів ===")
print(f"  Masив: {test_arr}")
print(f"  Bubble Sort swap: {count_swaps_bubble(test_arr)}")
print(f"  Selection Sort swap: {count_swaps_selection(test_arr)}  ← завжди ≤ n-1 = 19")

---

## Частина 3: Insertion Sort — сортування вставкою

---

### Інтуїція

Уявіть, що ви сортуєте карти в руці. Берете нову карту, порівнюєте з тими, що вже відсортовані зліва, і **вставляєте в потрібне місце**, зсуваючи інші вправо.

```
[ 9,  1, 15, 28,  6]
[ 1,  9, 15, 28,  6]   ← 1 < 9, вставляємо 1 на початок
[ 1,  9, 15, 28,  6]   ← 15 > 9, лишається
[ 1,  9, 15, 28,  6]   ← 28 > 15, лишається
[ 1,  6,  9, 15, 28]   ← 6 вставляємо між 1 та 9, зсув трьох елементів
```

### Складність

| Випадок | Складність | Причина |
|---------|-----------|--------|
| **Найкращий** | $O(n)$ | Масив вже відсортований — внутрішній while не виконується жодного разу |
| **Середній/Найгірший** | $O(n^2)$ | Кожен елемент зсуває всі попередні |

### Апаратна реальність

- **Cache locality:** Дуже висока — алгоритм працює з безперервним шматком пам'яті
- **Swaps/Shifts:** Багато зсувів пам'яті для великих масивів, але внутрішній цикл **миттєво переривається** на відсортованих ділянках
- **Стабільність:** **Стабільний** — не міняє порядок рівних елементів
- **Пам'ять:** $O(1)$ — in-place

### Реальне застосування

Ніхто не сортує ним великі дані. Але він є **базовим механізмом** всередині Merge Sort та Timsort — для сортування малих підмасивів (до 10–20 елементів), де його простота та відсутність накладних витрат рекурсії роблять його швидшим за будь-що інше.

---

In [ ]:
# ── Insertion Sort: класична реалізація ───────────────────────────────────────

def insertion_sort(arr):
    """
    Insertion Sort: для кожного елемента arr[i] знаходимо його правильну позицію
    у вже відсортованій лівій частині arr[0..i-1], зсуваючи більші елементи вправо.
    
    Ключова оптимізація: while loop — O(n) для майже відсортованих даних!
    """
    for i in range(1, len(arr)):
        current_val = arr[i]    # зберігаємо поточний елемент ("карта в руці")
        pos = i
        # Зсуваємо більші елементи вправо, щоб звільнити місце для current_val
        while pos > 0 and arr[pos - 1] > current_val:
            arr[pos] = arr[pos - 1]   # memory shift: копіюємо вліво-сусіда вправо
            pos -= 1
        arr[pos] = current_val        # вставляємо в правильну позицію
    return arr


# ── Демонстрація покрокового сортування ──────────────────────────────────────
print("=== Insertion Sort — покроково ===")
arr = [9, 1, 15, 28, 6]
print(f"  Початок: {arr}")

arr_demo = arr.copy()
for i in range(1, len(arr_demo)):
    current_val = arr_demo[i]
    pos = i
    while pos > 0 and arr_demo[pos - 1] > current_val:
        arr_demo[pos] = arr_demo[pos - 1]
        pos -= 1
    arr_demo[pos] = current_val
    print(f"  Крок {i}: вставили {current_val} → {arr_demo}")

print()
print("=== Insertion Sort: найкращий vs найгірший випадок ===")

def count_comparisons_insertion(arr):
    """Рахує кількість порівнянь у Insertion Sort."""
    arr = arr.copy()
    comparisons = 0
    for i in range(1, len(arr)):
        current_val = arr[i]
        pos = i
        while pos > 0:
            comparisons += 1
            if arr[pos - 1] > current_val:
                arr[pos] = arr[pos - 1]
                pos -= 1
            else:
                break
        arr[pos] = current_val
    return comparisons

n = 10
best_case  = list(range(n))          # вже відсортований: O(n)
worst_case = list(range(n, 0, -1))   # зворотний порядок: O(n^2)
print(f"  n = {n}")
print(f"  Найкращий (відсортований): {count_comparisons_insertion(best_case)} порівнянь  → O(n)")
print(f"  Найгірший (зворотний):     {count_comparisons_insertion(worst_case)} порівнянь  → O(n^2)")
print(f"  Теоретично O(n^2): {n*(n-1)//2}")

---

## Частина 4: Merge Sort — сортування злиттям

---

### Інтуїція

Стратегія **«розділяй і властвуй» (Divide & Conquer)**. Розрізаємо масив навпіл, доки не отримаємо підмасиви з одного елемента (які вже відсортовані). Потім попарно **«зливаємо»** їх назад у відсортованому порядку.

```
DIVIDE:   [120, 45, 68, 250, 176]
          [120, 45]  [68, 250, 176]
          [120] [45] [68] [250, 176]
                          [250] [176]

MERGE:    [45, 120]  [68] [176, 250]
          [45, 120]  [68, 176, 250]
          [45, 68, 120, 176, 250]  ← готово!
```

### Складність

| Випадок | Складність | Причина |
|---------|-----------|--------|
| **Всі** | $O(n \log n)$ | $\log n$ рівнів ділення × $O(n)$ на злиття кожного рівня |

### Апаратна реальність

- **Memory bandwidth:** Головний недолік. Під час злиття алгоритм **постійно виділяє нові масиви** → CPU ганяє байти між RAM і кешем → cache misses → повільніше за Quicksort на практиці
- **Рекурсія:** Глибока рекурсія $O(\log n)$ рівнів — накладні витрати на стек викликів
- **Стабільність:** **Стабільний** — злиття зберігає відносний порядок рівних елементів
- **Пам'ять:** $O(n)$ — потребує додаткову пам'ять для тимчасових масивів

### Реальне застосування

Ідеально для **Зв'язних списків (Linked Lists)** — злиття через перекидання вказівників не потребує копіювання пам'яті. Також для **зовнішнього сортування** (дані не влазять у RAM — читаємо чанками з диску).

---

In [ ]:
# ── Merge Sort: реалізація через merge функцію ────────────────────────────────

def merge(left_list, right_list):
    """
    Злиття двох відсортованих списків в один відсортований.
    Виділяє новий список sorted_list — O(n) пам'яті.
    Це і є вузьке місце Merge Sort з точки зору пам'яті.
    """
    sorted_list = []
    left_idx = right_idx = 0
    left_len, right_len = len(left_list), len(right_list)

    # Порівнюємо найменші елементи з обох сторін і додаємо менший
    while left_idx < left_len and right_idx < right_len:
        if left_list[left_idx] <= right_list[right_idx]:   # <= забезпечує стабільність!
            sorted_list.append(left_list[left_idx])
            left_idx += 1
        else:
            sorted_list.append(right_list[right_idx])
            right_idx += 1

    # Залишки одного зі списків (вже відсортовані)
    sorted_list.extend(left_list[left_idx:])
    sorted_list.extend(right_list[right_idx:])
    return sorted_list


def merge_sort(arr):
    """
    Merge Sort: рекурсивно ділимо на половини, потім зливаємо.
    Повертає НОВИЙ відсортований список (не in-place).
    """
    if len(arr) <= 1:
        return arr          # базовий випадок: 0 або 1 елемент — вже відсортований

    mid = len(arr) // 2     # ділимо навпіл
    left  = merge_sort(arr[:mid])   # рекурсивно сортуємо ліву половину
    right = merge_sort(arr[mid:])   # рекурсивно сортуємо праву половину

    return merge(left, right)       # зливаємо дві відсортовані половини


# ── Демонстрація ──────────────────────────────────────────────────────────────
print("=== Merge Sort ===")
data = [120, 45, 68, 250, 176]
print(f"  Вхід:            {data}")
result = merge_sort(data)
print(f"  Відсортовано:    {result}")

print()
print("=== Стабільність Merge Sort ===")
# Merge Sort зберігає порядок однакових елементів
records = [('Bob', 25), ('Alice', 25), ('Charlie', 22), ('Dave', 22)]
# Сортуємо за віком — Merge Sort гарантує: порядок (Bob, Alice) та (Charlie, Dave) збережеться
sorted_records = merge_sort(records)
print(f"  Вхід:  {records}")
print(f"  Вихід: {sorted(records, key=lambda x: x[1])}  ← рядки з однаковим віком в оригінальному порядку")

---

## Частина 5: Heap Sort — сортування через купу

---

### Інтуїція

Heap Sort структурує масив як **Max-Heap** (бінарне дерево, де батько завжди більший за дітей). Потім **$n$ разів** видаляє максимум: переміщує корінь (найбільший елемент) в кінець масиву, «відтинає» його від купи і відновлює властивість купи.

```
Max-Heap зі [35, 12, 43, 8, 51]:

         51          ← індекс 0 (корінь)
        /  \
      12    43       ← індекси 1, 2
     /  \
    8   35           ← індекси 3, 4

Формула вузлів: left = 2*i+1, right = 2*i+2, parent = (i-1)//2
```

### Складність

| Операція | Складність | Причина |
|---------|-----------|--------|
| `heapify` (побудова купи) | $O(n)$ | Амортизований аналіз: нижні рівні дешевше |
| `sink` (відновлення купи) | $O(\log n)$ | Висота дерева $\log n$ |
| **Загальна** | **$O(n \log n)$** | $n$ видалень × $O(\log n)$ кожне |

### Апаратна реальність

- **Cache locality:** Погана — «стрибки» між вузлами дерева порушують cache-лінії CPU → на практиці повільніший за Quicksort
- **Пам'ять:** $O(1)$ — in-place! Купа зберігається в оригінальному масиві через формули індексів
- **Стабільність:** **Нестабільний** — далекі swap через дерево руйнують порядок рівних елементів
- **Гарантований $O(n \log n)$** — немає найгіршого випадку $O(n^2)$ як у Quicksort

### Реальне застосування

**Алгоритм Introsort** (C++ `std::sort`) автоматично перемикається на Heap Sort, коли рекурсія Quicksort стає занадто глибокою — це гарантує $O(n \log n)$ у найгіршому випадку.

---

In [ ]:
# ── Heap Sort ─────────────────────────────────────────────────────────────────

def heapify(arr, heap_size, root_index):
    """
    heapify: відновлює властивість Max-Heap для піддерева з коренем root_index.
    heap_size визначає логічний кінець купи в масиві.
    O(log n) — спускається по висоті дерева.
    """
    largest = root_index             # припускаємо: корінь — найбільший
    left_child  = 2 * root_index + 1  # індекс лівого нащадка
    right_child = 2 * root_index + 2  # індекс правого нащадка

    # Якщо лівий нащадок існує і більший за поточний largest
    if left_child < heap_size and arr[left_child] > arr[largest]:
        largest = left_child

    # Якщо правий нащадок існує і більший за поточний largest
    if right_child < heap_size and arr[right_child] > arr[largest]:
        largest = right_child

    # Якщо найбільший — не корінь, swap і рекурсивно відновлюємо нижче
    if largest != root_index:
        arr[root_index], arr[largest] = arr[largest], arr[root_index]
        heapify(arr, heap_size, largest)   # після swap нижнє піддерево може порушитись


def heap_sort(arr):
    """
    Heap Sort: два фази.
    Фаза 1: побудова Max-Heap з масиву (O(n)).
    Фаза 2: n разів виймаємо максимум з купи і кладемо в кінець масиву.
    """
    n = len(arr)

    # ── Фаза 1: побудова Max-Heap ─────────────────────────────────────────────
    # Починаємо з останнього не-листкового вузла (n//2 - 1) і йдемо вгору
    # Листки вже є валідними купами розміру 1
    for i in range(n // 2 - 1, -1, -1):  # від n//2-1 до 0 включно
        heapify(arr, n, i)

    # ── Фаза 2: витягування елементів з купи ─────────────────────────────────
    for i in range(n - 1, 0, -1):
        arr[0], arr[i] = arr[i], arr[0]   # swap: максимум (корінь) → кінець масиву
        heapify(arr, i, 0)                 # відновлюємо купу (без останнього елемента)

    return arr


# ── Демонстрація ──────────────────────────────────────────────────────────────
print("=== Heap Sort ===")
data = [35, 12, 43, 8, 51]
print(f"  Вхід:            {data}")
result = heap_sort(data.copy())
print(f"  Відсортовано:    {result}")

print()
print("=== heapq: вбудована Python Min-Heap ===")
import heapq

def heap_sort_builtin(arr):
    """Timsort Python не використовує, але heapq дозволяє heapsort через min-heap."""
    heap = arr.copy()
    heapq.heapify(heap)              # O(n) — перетворює список в min-heap in-place
    return [heapq.heappop(heap) for _ in range(len(heap))]  # O(n log n)

data = [35, 12, 43, 8, 51]
print(f"  heapq result: {heap_sort_builtin(data)}")
print(f"  Наш heap_sort: {heap_sort(data.copy())}")
print(f"  Результати однакові: {heap_sort_builtin(data) == sorted(data)}")

---

## Частина 6: Quick Sort — швидке сортування

---

### Інтуїція

Вибираємо **«опорний» елемент (pivot)**. Елементи менші за pivot переміщуємо ліворуч, більші — праворуч. Далі рекурсивно застосовуємо до обох частин.

```
[22, 5, 1, 18, 99]   pivot = 18 (середній)
 ↓
[ 5,  1, 18, 22, 99]  ← все < 18 ліворуч,  > 18 праворуч
         ↑ pivot на місці!

Далі рекурсивно:  [5, 1] і [22, 99]
[1, 5, 18, 22, 99]   ← готово!
```

### Складність

| Випадок | Складність | Причина |
|---------|-----------|--------|
| **Найкращий/Середній** | $O(n \log n)$ | Збалансований розподіл → $\log n$ рівнів рекурсії |
| **Найгірший** | $O(n^2)$ | Відсортований масив + pivot = перший елемент → незбалансований розподіл |

### Апаратна реальність: чому Quicksort найшвидший на практиці

- **Король кешу (Cache locality):** Фаза розбиття (partitioning) сканує масив **лінійно** зліва направо та справа наліво. Ці дані вже в кеш-лінії CPU → мінімум cache misses
- **In-place:** Жодного виділення додаткової пам'яті для масивів, тільки $O(\log n)$ на стек рекурсії
- **Comparisons vs Swaps:** Багато порівнянь, мінімум swap. CPU може порівнювати значення прямо в регістрах — це набагато дешевше, ніж фізично переписувати комірки RAM
- **Стабільність:** **Нестабільний** — далекі swap через pivot руйнують порядок рівних елементів

### Реальне застосування

`qsort` в С-стандартній бібліотеці, `std::sort` у C++ (як частина Introsort). Основа більшості системних сортувань.

---

In [ ]:
# ── Quick Sort: схема Hoare (класична для співбесід) ──────────────────────────

def partition_lomuto(arr, low, high):
    """
    Partition схема Lomuto: pivot = arr[high].
    Два вказівники: i (межа малих елементів) і j (поточний).
    Простіша для розуміння, але робить більше swap ніж Hoare.
    """
    pivot = arr[high]              # обираємо pivot як останній елемент
    i = low - 1                    # i — «права межа" малих елементів
    for j in range(low, high):
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr[i]   # in-place swap!
    arr[i + 1], arr[high] = arr[high], arr[i + 1]   # ставимо pivot на місце
    return i + 1                   # індекс pivot після розбиття


def partition_hoare(arr, low, high):
    """
    Partition схема Hoare: pivot = середній елемент.
    Два вказівники з'їжджаються назустріч — ефективніше!
    Менше swap ніж Lomuto, краща cache locality.
    """
    pivot = arr[(low + high) // 2]  # середній елемент — менш вразливий до O(n^2)
    i = low - 1
    j = high + 1
    while True:
        i += 1
        while arr[i] < pivot:        # рухаємо i вправо поки елемент < pivot
            i += 1
        j -= 1
        while arr[j] > pivot:        # рухаємо j вліво поки елемент > pivot
            j -= 1
        if i >= j:
            return j                 # j — індекс розбиття
        arr[i], arr[j] = arr[j], arr[i]  # in-place swap двох «неправильних» елементів


def quick_sort_lomuto(arr, low=None, high=None):
    """Quick Sort з Lomuto partition."""
    if low is None: low = 0
    if high is None: high = len(arr) - 1
    if low < high:
        pi = partition_lomuto(arr, low, high)
        quick_sort_lomuto(arr, low, pi - 1)
        quick_sort_lomuto(arr, pi + 1, high)


def quick_sort(arr):
    """Quick Sort через внутрішню функцію — зовнішній API без low/high."""
    def _qs(items, low, high):
        if low < high:
            split = partition_hoare(items, low, high)
            _qs(items, low, split)        # ліва частина [low .. split]
            _qs(items, split + 1, high)   # права частина [split+1 .. high]
    _qs(arr, 0, len(arr) - 1)


# ── Демонстрація ──────────────────────────────────────────────────────────────
print("=== Quick Sort ===")
data = [22, 5, 1, 18, 99]
print(f"  Вхід: {data}")

arr1 = data.copy()
quick_sort_lomuto(arr1)
print(f"  Lomuto partition: {arr1}")

arr2 = data.copy()
quick_sort(arr2)
print(f"  Hoare partition:  {arr2}")

print()
print("=== Демонстрація найгіршого випадку O(n^2) ===")
import sys

# Вже відсортований масив + Lomuto pivot=last → O(n^2)
n = 100
sorted_arr = list(range(n))

# Ітеративна версія для уникнення RecursionError
def quick_sort_iterative(arr):
    """Ітеративний Quick Sort (Lomuto) без рекурсії."""
    stack = [(0, len(arr) - 1)]
    while stack:
        low, high = stack.pop()
        if low < high:
            pi = partition_lomuto(arr, low, high)
            stack.append((low, pi - 1))
            stack.append((pi + 1, high))

arr3 = sorted_arr.copy()
quick_sort_iterative(arr3)
print(f"  Відсортований масив з {n} елементів — Lomuto+ітерація: OK {arr3[:5]}...")
print(f"  (Рекурсивна версія впала б в RecursionError при великих n!)") 

---

## Частина 7: Timsort — секретна зброя Python

---

### Інтуїція

**Гібридний алгоритм**, винайдений Тімом Пітерсом у 2002 році для CPython. Він помітив, що **реальні дані рідко бувають повністю хаотичними** — в них часто є вже відсортовані підмасиви (runs).

```
Алгоритм Timsort:

1. Сканувати масив на «runs» (відсортовані послідовності)
   [3, 5, 8, 2, 4, 9, 1, 7]  →  runs: [3,5,8] [2,4,9] [1,7]

2. Якщо run коротший за MIN_RUN (~32-64) → Insertion Sort до мін розміру
   Insertion Sort: O(1) simple ops, no recursion, perfect cache locality

3. Merge runs попарно за допомогою Merge Sort
   [2,3,4,5,8,9] + [1,7] → [1,2,3,4,5,7,8,9]
```

### Складність

| Випадок | Складність | Причина |
|---------|-----------|--------|
| **Найкращий** | $O(n)$ | Дані вже відсортовані → один run, Insertion Sort = $O(n)$ |
| **Середній/Найгірший** | $O(n \log n)$ | Merge Sort гарантує |

### Апаратна реальність

- Поєднує $O(1)$ локальність і **швидкість Insertion Sort** для малих масивів
- з надійним $O(n \log n)$ **Merge Sort** для великих
- Реалізований на рівні C в CPython — мінімальні overhead'и
- **Стабільний** — критично для Python, щоб `sorted(records, key=age)` зберігав порядок після `sorted(records, key=name)`
- **Пам'ять:** $O(n)$ — як Merge Sort

### Реальне застосування

- **Python:** `list.sort()` та `sorted()` — завжди Timsort
- **Java:** `Arrays.sort()` для об'єктів
- **Android:** `java.util.Arrays.sort()`

---

In [ ]:
# ── Timsort: використання вбудованого API та демонстрація стабільності ─────────

print("=== Timsort: list.sort() та sorted() ===")

# list.sort() — in-place, повертає None
apples = [2, 1, 1, 3, 1, 2, 2]
apples.sort()                        # Timsort in-place
print(f"  list.sort() in-place:  {apples}")

# sorted() — повертає новий список
nums = [2, 1, 1, 3, 1, 2, 2]
sorted_nums = sorted(nums)           # Timsort, новий список
print(f"  sorted() новий список: {sorted_nums}")

# Зворотне сортування
desc = sorted(nums, reverse=True)
print(f"  reverse=True:          {desc}")

print()
print("=== Стабільність Timsort: багаторівневе сортування ===")
# Реальний сценарій: logs вже відсортовані за timestamp
# Сортуємо за error_code — стабільність гарантує збереження timestamp-порядку
logs = [
    {'code': 500, 'time': '10:00', 'msg': 'first 500'},
    {'code': 200, 'time': '10:05', 'msg': 'ok'},
    {'code': 500, 'time': '10:10', 'msg': 'second 500'},
    {'code': 404, 'time': '10:15', 'msg': 'not found'},
]

sorted_logs = sorted(logs, key=lambda x: x['code'])
print(f"  Сортування за error code — 500 помилки залишились у chronological order:")
for log in sorted_logs:
    print(f"    code={log['code']}, time={log['time']}: {log['msg']}")

print()
print("=== Timsort: O(n) для вже відсортованих даних ===")
import timeit

n = 100_000
sorted_data   = list(range(n))
reversed_data = list(range(n, 0, -1))
random_data   = sorted_data.copy()
import random; random.seed(42); random.shuffle(random_data)

t_sorted   = timeit.timeit(lambda: sorted(sorted_data),   number=50) / 50 * 1000
t_reversed = timeit.timeit(lambda: sorted(reversed_data), number=50) / 50 * 1000
t_random   = timeit.timeit(lambda: sorted(random_data),   number=50) / 50 * 1000

print(f"  n={n}, середній час (мс):")
print(f"  Відсортований:   {t_sorted:.2f} мс  ← O(n) — майже нуль!")
print(f"  Зворотний:       {t_reversed:.2f} мс  ← O(n) теж — Timsort виявляє descending run")
print(f"  Випадковий:      {t_random:.2f} мс  ← O(n log n)")

---

## Частина 8: Апаратна реальність — кеш, пам'ять, рекурсія

---

### CPU Cache: чому Quicksort насправді найшвидший

Big-O описує **кількість операцій**. Але не всі операції однакові за часом:

```
Ієрархія пам'яті (затримка доступу):

CPU Register    │ ~0.25 нс  │ ███████████████████████████████ (1x)
L1 Cache (32KB) │ ~1 нс     │ ████ (4x)
L2 Cache (256KB)│ ~5 нс     │ ████████████████████ (20x)
L3 Cache (8MB)  │ ~20 нс    │ ████████████████████████████████████████ (80x)
RAM             │ ~100 нс   │ ████████████████████...400x (!)  │
SSD             │ ~100 мкс  │  400,000x
```

Коли алгоритм **лінійно сканує** масив (як Quicksort's partitioning), CPU підвантажує цілу кеш-лінію (64 байти = 16 int32) за 1 доступ до RAM — і наступні 15 звернень безкоштовні.

Коли алгоритм **стрибає** по масиву (як Heap Sort по індексах дерева), кожен «стрибок» — потенційний cache miss → доступ до RAM → 400x повільніше.

### Recursion Overhead: чому рекурсія не безкоштовна

Кожен рекурсивний виклик:
1. Зберігає поточний стан (локальні змінні + адрес повернення) у **Call Stack**
2. Python Call Stack обмежений ~1000 кадрів за замовчуванням
3. Quicksort: глибина $O(\log n)$ → для $10^9$ елементів лише **30 кадрів** — безпечно
4. Вироджений Quicksort на відсортованих даних: глибина $O(n)$ → **RecursionError**

### Memory Allocation: прихована вартість Merge Sort

```python
# Merge Sort на кожному злитті виконує:
sorted_list = []           # malloc: виділення пам'яті
sorted_list.append(...)    # запис
return sorted_list         # передача в батьківський кадр
```

На $n = 10^6$: Merge Sort виділяє приблизно $n \log n \approx 20 \cdot 10^6$ байт тимчасових списків. Ці виділення та звільнення пам'яті — **реальний bottleneck** на великих даних.

---

---

## Частина 9: Benchmark — вимірюємо час усіх алгоритмів

---

Натхненний статтею Barack Obama Google Interview: *"Bubble Sort is the wrong way to go"* — давайте перевіримо це числами!

Ми відтворимо таблицю з реального бенчмарку:

| Run | Bubble | Selection | Insertion | Heap | Merge | Quick |
|-----|--------|-----------|-----------|------|-------|-------|
| Avg | 5.085s | 1.247s | 1.610s | 0.042s | 0.028s | 0.017s |

---

In [ ]:
# ── Benchmark: порівнюємо час виконання всіх алгоритмів ─────────────────────
import timeit
import random

# Генеруємо тестовий масив: 5000 випадкових чисел від 0 до 1000
N = 5000
RUNS = 3   # кількість запусків (оригінал: 10, зменшено для швидшої демонстрації)

def generate_test_data():
    return [random.randint(0, 1000) for _ in range(N)]

# Всі алгоритми, що сортують in-place (змінюють масив, повертають None/arr)
def run_bubble(arr):    return bubble_sort(arr)
def run_selection(arr): return selection_sort(arr)
def run_insertion(arr): return insertion_sort(arr)
def run_heap(arr):      return heap_sort(arr)
def run_merge(arr):     arr[:] = merge_sort(arr); return arr
def run_quick(arr):     quick_sort(arr); return arr
def run_timsort(arr):   arr.sort(); return arr

algorithms = [
    ('Bubble Sort',    run_bubble),
    ('Selection Sort', run_selection),
    ('Insertion Sort', run_insertion),
    ('Heap Sort',      run_heap),
    ('Merge Sort',     run_merge),
    ('Quick Sort',     run_quick),
    ('Timsort',        run_timsort),
]

results = {name: [] for name, _ in algorithms}

print(f"Benchmark: n={N} елементів, {RUNS} запусків кожного алгоритму")
print(f"{'='*65}")
print(f"{'Алгоритм':<20} | {'Run 1':>8} | {'Run 2':>8} | {'Run 3':>8} | {'Avg':>8}")
print(f"{'-'*65}")

for name, func in algorithms:
    times = []
    for _ in range(RUNS):
        data = generate_test_data()
        t = timeit.timeit(lambda d=data: func(d), number=1)
        times.append(t)
    results[name] = times
    avg = sum(times) / len(times)
    times_str = ' | '.join(f'{t:>8.5f}' for t in times)
    print(f"{name:<20} | {times_str} | {avg:>8.5f}")

print(f"{'='*65}")
print(f"  (час у секундах)")

# Підсумок: у скільки разів кожен алгоритм повільніший за Timsort
print()
print("=== Відносна швидкість (vs Timsort) ===")
timsort_avg = sum(results['Timsort']) / RUNS
for name, _ in algorithms:
    avg = sum(results[name]) / RUNS
    ratio = avg / timsort_avg
    bar = '█' * min(int(ratio * 2), 60)
    print(f"  {name:<20}: {ratio:>6.1f}x  {bar}")

---

## Частина 10: Питання для співбесіди

---

### Питання 1: $O(n^2)$ проти $O(n \log n)$ — коли $O(n^2)$ краще?

> **Питання:** «Порівняйте $O(n^2)$ алгоритми з $O(n \log n)$. Коли ви б обрали $O(n^2)$?»

**Коротка відповідь:** $O(n \log n)$ алгоритми використовують «розділяй і властвуй», різко скорочуючи кількість кроків при зростанні $n$. Але Insertion Sort $O(n^2)$ є кращим вибором для дуже малих масивів (до ~20 елементів) або майже відсортованих даних, де його найкращий випадок $O(n)$ і відсутність накладних витрат рекурсії дають реальну перевагу.

**Типова помилка:** *«$O(n^2)$ завжди погано»* — кандидати не розуміють, що Timsort сам використовує Insertion Sort всередині!

---

### Питання 2: In-place vs Out-of-place — чому Quicksort кращий за Merge Sort на практиці?

> **Питання:** «Обидва мають середню $O(n \log n)$. Чому Quicksort зазвичай швидший?»

**Коротка відповідь:** Merge Sort — out-of-place і потребує $O(n)$ пам'яті для злиття. Quicksort — in-place, сортує безпосередньо в оригінальному масиві, потребуючи лише $O(\log n)$ на стек рекурсії. Оскільки Quicksort мінімізує виділення пам'яті та має чудову локальність кешу (лінійне сканування), він переважає Merge Sort на звичайних масивах.

**Типова помилка:** Вважати, що Quicksort гарантує $O(n \log n)$ — якщо відсортований масив + pivot=перший елемент → $O(n^2)$.

---

### Питання 3: Стабільність сортування

> **Питання:** «Що означає 'стабільний' алгоритм? Навіщо це важливо?»

**Коротка відповідь:** Стабільний алгоритм гарантує, що елементи з однаковим ключем зберігають свій відносний порядок після сортування. Це критично при **ланцюговому сортуванні**: спочатку за `timestamp`, потім за `error_code` — стабільне сортування гарантує, що серед однакових `error_code` порядок `timestamp` збережеться.

| Алгоритм | Стабільний? |
|---------|------------|
| Bubble Sort | ✅ Так |
| Insertion Sort | ✅ Так |
| Merge Sort | ✅ Так |
| Timsort | ✅ Так |
| Selection Sort | ❌ Ні |
| Quick Sort | ❌ Ні |
| Heap Sort | ❌ Ні |

**Типова помилка:** Плутати «стабільний» з «надійний» або «передбачуваний за часом».

---

### Питання 4: Heap Sort внутрішній механізм

> **Питання:** «Як Heap Sort досягає $O(n \log n)$ без Divide & Conquer рекурсії Merge/Quick Sort?»

**Коротка відповідь:** Heap Sort перетворює масив у Max-Heap (батько завжди більший за дітей). Потім $n$ разів: swap(корінь, кінець) + `heapify`. Відновлення купи займає $O(\log n)$ (висота дерева), $n$ ітерацій → $O(n \log n)$. Все в оригінальному масиві — $O(1)$ пам'яті.

**Типова помилка:** Думати, що Heap Sort потребує $O(n)$ пам'яті на дерево з вказівниками — ні, купа зберігається в плоскому масиві через формули індексів!

---

### Питання 5: Чому Python використовує Timsort, а не Quicksort?

> **Питання:** «Коли ви викликаєте `list.sort()` в Python, який алгоритм виконується?»

**Коротка відповідь:** Python використовує **Timsort** — стабільний гібрид Merge Sort + Insertion Sort, що оптимально використовує частково відсортовані «runs» у реальних даних. Python не може використовувати Quicksort через його нестабільність і ризик $O(n^2)$ на відсортованих даних.

**Типова помилка:** «Python використовує Quicksort» — поширений міф.

---

In [ ]:
# ── Демонстрація питань для співбесіди ─────────────────────────────────────────

print("=== Q1: Insertion Sort vs Merge Sort на малих масивах ===")
import timeit

SMALL_N = 15
REPS = 10_000

small_data = [random.randint(0, 100) for _ in range(SMALL_N)]

t_insertion = timeit.timeit(lambda: insertion_sort(small_data.copy()), number=REPS) / REPS * 1e6
t_merge     = timeit.timeit(lambda: merge_sort(small_data.copy()),     number=REPS) / REPS * 1e6
t_timsort   = timeit.timeit(lambda: sorted(small_data),                number=REPS) / REPS * 1e6

print(f"  n={SMALL_N} (мікросекунди):")
print(f"  Insertion Sort: {t_insertion:.2f} мкс")
print(f"  Merge Sort:     {t_merge:.2f} мкс")
print(f"  Timsort:        {t_timsort:.2f} мкс  ← Timsort обирає Insertion Sort внутрішньо!")

print()
print("=== Q3: Демонстрація стабільності ===")
# Нестабільний Selection Sort може зруйнувати порядок
data_with_keys = [(25, 'Bob'), (22, 'Charlie'), (25, 'Alice'), (22, 'Dave')]

# Timsort (стабільний): збереже порядок Bob перед Alice (обидва age=25)
stable = sorted(data_with_keys, key=lambda x: x[0])
print(f"  Вхід:   {data_with_keys}")
print(f"  Timsort (стабільний): {stable}")
print(f"  Bob залишився перед Alice? {'Yes ✓' if stable[2][1] == 'Bob' else 'No ✗'}")

print()
print("=== Q4: Python RecursionError на виродженому Quick Sort ===")
import sys
print(f"  Python recursion limit: {sys.getrecursionlimit()} кадрів")
print(f"  Відсортований масив n=5000 + recursive Quick Sort з pivot=last → RecursionError")
print(f"  Рішення: ітеративний Quick Sort або random pivot (median-of-3)")

print()
print("=== Q5: Python built-in sort — це Timsort ===")
print(f"  type(list.sort)  = {type([].sort)}")
import inspect
# Timsort реалізований у C в CPython — тому inspect.getsource не доступний
print(f"  Реалізований у C (Objects/listobject.c) — через Timsort Tim Peters 2002")
print(f"  sorted([]) — також Timsort, але повертає новий list")

---

## Частина 11: Підсумок

---

### Велика таблиця алгоритмів

| Алгоритм | Best | Average | Worst | Пам'ять | Стабільний | Реальне застосування |
|---------|------|---------|-------|---------|-----------|---------------------|
| **Bubble Sort** | $O(n)$ | $O(n^2)$ | $O(n^2)$ | $O(1)$ | ✅ | Навчання. Ніде в продакшені |
| **Selection Sort** | $O(n^2)$ | $O(n^2)$ | $O(n^2)$ | $O(1)$ | ❌ | Мінімум swap — коли swap дорогий |
| **Insertion Sort** | $O(n)$ | $O(n^2)$ | $O(n^2)$ | $O(1)$ | ✅ | Малі масиви (<20), всередині Timsort |
| **Merge Sort** | $O(n \log n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(n)$ | ✅ | Linked Lists, зовнішнє сортування |
| **Heap Sort** | $O(n \log n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(1)$ | ❌ | Fallback у Introsort (C++ std::sort) |
| **Quick Sort** | $O(n \log n)$ | $O(n \log n)$ | $O(n^2)$ | $O(\log n)$ | ❌ | C qsort, загальне сортування масивів |
| **Timsort** | $O(n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(n)$ | ✅ | Python sort(), Java Arrays.sort() |

### Ключові архітектурні правила

1. **Big-O — лише гарантія масштабу.** Уникайте $O(n^2)$ для великих $n$, але реальна битва виграється апаратом
2. **In-place = кеш-дружній.** Quicksort перемагає Merge Sort на масивах завдяки cache locality
3. **Стабільність = передбачуваність.** Для Python, бізнес-логіки, ланцюгових сортувань — завжди обирайте стабільний алгоритм
4. **Малі масиви = Insertion Sort.** Timsort це знає і використовує
5. **Ніколи `list.pop(0)` у циклі** — це $O(n^2)$, використовуйте `deque.popleft()`

### Алгоритм вибору структури

```
Що потрібно сортувати?
         │
    ┌────┴────────────────────────────┐
    │                                 │
Python об'єкти / потрібна стабільність    C/системний рівень
         │                                 │
    list.sort() / sorted()             Quick Sort / Introsort
    (Timsort — завжди!)               (C qsort / std::sort)

Спеціальні випадки:
  Linked List          → Merge Sort
  Зовнішнє сортування  → Merge Sort (чанки з диску)
  Майже відсортоване   → Insertion Sort або Timsort
  Потрібен O(1) пам'ять + O(n log n) гарантія → Heap Sort
```

---
*Наступний урок: Динамічне програмування*